In [96]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("uciml/pima-indians-diabetes-database")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\Acer\.cache\kagglehub\datasets\uciml\pima-indians-diabetes-database\versions\1


In [97]:
import pandas as pd
df = pd.read_csv("diabetes.csv")

In [98]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [99]:
x = df.iloc[: , :-1].values 
y = df.iloc[: , -1].values

In [100]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [101]:
x = scaler.fit_transform(x)

In [102]:
x.shape

(768, 8)

In [103]:
from sklearn.model_selection import train_test_split 
x_train , x_test , y_train , y_test = train_test_split(x, y , test_size= 0.2 , random_state= 42)

In [132]:
import numpy as np
import matplotlib.pyplot as plt 
import tensorflow 
from tensorflow import keras 
from keras.layers import Dense , Dropout
from keras.models import Sequential
from keras import Input

In [105]:
model = Sequential()

model.add(Dense(32 , input_dim = 8 , activation= 'relu'))
model.add(Dense(1 , activation= 'sigmoid'))

model.compile(optimizer= 'adam' , loss= 'binary_crossentropy' , metrics= ['accuracy'])

c:\Users\Acer\anaconda3\envs\santoshenv\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [106]:
model.fit(x_train , y_train , validation_data= (x_test , y_test) , batch_size= 32 , epochs= 100 , verbose= False)

In [107]:
# 1. How to select appropriate optimizer ?
# 2. No. of nodes in layer 
# 3. How to select no.layers 
# 4. All in all over model 

In [108]:
import keras_tuner as kt 

In [109]:
def build_model(hp):

    model = Sequential([Input(shape= (8,))])

    model.add(Dense(32, activation='relu'))
    model.add(Dense(1 , activation= 'sigmoid'))

    optimizer = hp.Choice('optimizer' , values = ['adam' , 'rmsprop' , 'sgd' , 'adadelta'])
    model.compile(optimizer= optimizer , loss= 'binary_crossentropy' , metrics= ['accuracy'])
    
    return model

In [110]:
tuner = kt.RandomSearch(
    build_model , 
    objective = 'val_accuracy',
    max_trials = 5,
    project_name='optimizer_tuning',
    directory='tuner_results'
    )

In [111]:
tuner.search(x_train , y_train , validation_data = (x_test , y_test) , epochs = 5)

Trial 4 Complete [00h 00m 02s]
val_accuracy: 0.7077922224998474

Best val_accuracy So Far: 0.7662337422370911
Total elapsed time: 00h 00m 09s


In [112]:
tuner.results_summary()

Results summary
Results in tuner_results\optimizer_tuning
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 0 summary
Hyperparameters:
optimizer: rmsprop
Score: 0.7662337422370911

Trial 1 summary
Hyperparameters:
optimizer: adam
Score: 0.7532467246055603

Trial 3 summary
Hyperparameters:
optimizer: sgd
Score: 0.7077922224998474

Trial 2 summary
Hyperparameters:
optimizer: adadelta
Score: 0.3896103799343109


In [113]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'rmsprop'}

In [114]:
model = tuner.get_best_models(num_models= 1)[0]

c:\Users\Acer\anaconda3\envs\santoshenv\Lib\site-packages\keras\src\saving\saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [115]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [116]:
model.fit(x_train , y_train , batch_size= 32 , epochs= 100 , initial_epoch= 6 , validation_data= (x_test , y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.7394 - loss: 0.5183 - val_accuracy: 0.7662 - val_loss: 0.5125
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7443 - loss: 0.5029 - val_accuracy: 0.7727 - val_loss: 0.5055
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7557 - loss: 0.4930 - val_accuracy: 0.7662 - val_loss: 0.5013
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7557 - loss: 0.4847 - val_accuracy: 0.7727 - val_loss: 0.4976
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7541 - loss: 0.4778 - val_accuracy: 0.7597 - val_loss: 0.4951
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7622 - loss: 0.4727 - val_accuracy: 0.7727 - val_loss: 0.4933
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7638 - loss: 0.4676 - val_accuracy: 0.7597 - val_loss: 0.4921
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7655 - loss: 0.4637 - val_accuracy: 0.76

In [117]:
def build_model2(hp):
    model2 = Sequential()

    units = hp.Int('units' , min_value = 8 , max_value = 128 , step= 8)

    model2.add(Dense(units = units , activation= 'relu' , input_dim = 8))
    model2.add(Dense(1 , activation= 'sigmoid'))

    model2.compile(optimizer= 'rmsprop' , loss = 'binary_crossentropy' , metrics= ['accuracy'])

    return model2 

In [118]:
tuner2 = kt.RandomSearch(build_model2 , objective= 'val_accuracy' , max_trials= 5, project_name='units_tuning', directory='tuner2_results')

In [119]:
tuner2.search(x_train , y_train , epochs = 5 , validation_data = (x_test , y_test))

Trial 5 Complete [00h 00m 02s]
val_accuracy: 0.7597402334213257

Best val_accuracy So Far: 0.7597402334213257
Total elapsed time: 00h 00m 10s


In [120]:
tuner2.get_best_hyperparameters()[0].values

{'units': 88}

In [121]:
model2= tuner2.get_best_models(num_models= 1)[0]

c:\Users\Acer\anaconda3\envs\santoshenv\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\Acer\anaconda3\envs\santoshenv\Lib\site-packages\keras\src\saving\saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [123]:
model2.fit(x_train , y_train , batch_size= 32 , epochs = 100 , initial_epoch= 6 , validation_data= (x_test , y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.7590 - loss: 0.4815 - val_accuracy: 0.7727 - val_loss: 0.5092
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7736 - loss: 0.4686 - val_accuracy: 0.7532 - val_loss: 0.5043
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7834 - loss: 0.4623 - val_accuracy: 0.7597 - val_loss: 0.5029
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7834 - loss: 0.4575 - val_accuracy: 0.7597 - val_loss: 0.5042
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7850 - loss: 0.4532 - val_accuracy: 0.7597 - val_loss: 0.5053
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7834 - loss: 0.4501 - val_accuracy: 0.7597 - val_loss: 0.5058
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7850 - loss: 0.4467 - val_accuracy: 0.7597 - val_loss: 0.5073
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7801 - loss: 0.4441 - val_accuracy: 0.75

In [124]:
def build_model(hp):
    model = Sequential()

    model.add(Dense(88 , activation='relu', input_dim=8))

    for i in range(hp.Int('num_layers' , min_value = 1 , max_value = 10)):
        model.add(Dense(88 , activation= 'relu'))

    model.add(Dense(1 , activation= 'sigmoid'))

    model.compile(optimizer='rmsprop',
                   loss='binary_crossentropy', metrics=['accuracy'])

    return model

In [125]:
tuner = kt.RandomSearch(build_model , objective= 'val_accuracy' ,max_trials= 5 , project_name = 'numlayers' , directory = 'layerfinding')

c:\Users\Acer\anaconda3\envs\santoshenv\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [127]:
tuner.search(x_train , y_train , validation_data = (x_test , y_test) , epochs = 5)

Trial 5 Complete [00h 00m 03s]
val_accuracy: 0.7857142686843872

Best val_accuracy So Far: 0.798701286315918
Total elapsed time: 00h 00m 54s


In [129]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 4}

In [130]:
model = tuner.get_best_models(num_models= 1)[0]

c:\Users\Acer\anaconda3\envs\santoshenv\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\Acer\anaconda3\envs\santoshenv\Lib\site-packages\keras\src\saving\saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [131]:
model.fit(x_train , y_train , validation_data= (x_test , y_test) , epochs= 100 , initial_epoch= 6)

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.7720 - loss: 0.4717 - val_accuracy: 0.7792 - val_loss: 0.5012
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7818 - loss: 0.4421 - val_accuracy: 0.7792 - val_loss: 0.5205
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7850 - loss: 0.4344 - val_accuracy: 0.7597 - val_loss: 0.5166
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7915 - loss: 0.4243 - val_accuracy: 0.7727 - val_loss: 0.5231
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8046 - loss: 0.4119 - val_accuracy: 0.7792 - val_loss: 0.5124
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7980 - loss: 0.4104 - val_accuracy: 0.6883 - val_loss: 0.5904
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8094 - loss: 0.3968 - val_accuracy: 0.7727 - val_loss: 0.5944
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8160 - loss: 0.3850 - val_accuracy: 0.76

In [148]:
# last main model 
def build_model(hp):
    model = Sequential()
    counter = 0

    for i in range(hp.Int('numlayers' , min_value = 1 , max_value = 5)):  

        if counter == 0:
            model.add(Dense(units= hp.Int('layer'+ str(i) , min_value = 16 , max_value = 128 , step = 16) , activation= hp.Choice('activation'+str(i) , values = ['relu' , 'tanh']) , input_dim = 8))
            model.add(Dropout(hp.Choice('dropout'+str(i) , values = [0.1 , 0.2 , 0.3])))  

        else:
            model.add(Dense(units= hp.Int('layer' + str(i) , min_value = 16 , max_value = 128 , step = 16) , activation= hp.Choice('activation'+str(i) , values = ['relu' , 'tanh'])))
            model.add(Dropout(hp.Choice('dropout'+str(i), values=[0.1, 0.2, 0.3])))  
            
        counter += 1

    model.add(Dense(1, activation='sigmoid')) 

    model.compile(optimizer= hp.Choice('optimizer' , values = ['adam' , 'rmsprop']) , loss= 'binary_crossentropy' , metrics= ['accuracy'])

    return model

In [149]:
tuner = kt.RandomSearch(build_model , objective= 'val_accuracy' , max_trials= 10, directory = 'final' , project_name = 'final_result', overwrite=True)

c:\Users\Acer\anaconda3\envs\santoshenv\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [150]:
tuner.search(x_train , y_train , validation_data = (x_test , y_test) , epochs = 20, verbose=1) 

Trial 10 Complete [00h 00m 05s]
val_accuracy: 0.7857142686843872

Best val_accuracy So Far: 0.7857142686843872
Total elapsed time: 00h 00m 44s


In [151]:
tuner.get_best_hyperparameters()[0].values

{'numlayers': 1,
 'layer0': 48,
 'activation0': 'tanh',
 'dropout0': 0.3,
 'optimizer': 'rmsprop',
 'layer1': 112,
 'activation1': 'tanh',
 'dropout1': 0.3,
 'layer2': 16,
 'activation2': 'tanh',
 'dropout2': 0.3,
 'layer3': 128,
 'activation3': 'relu',
 'dropout3': 0.2,
 'layer4': 32,
 'activation4': 'tanh',
 'dropout4': 0.1}

In [152]:
model = tuner.get_best_models(num_models= 1)[0]

c:\Users\Acer\anaconda3\envs\santoshenv\Lib\site-packages\keras\src\saving\saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [153]:
model.fit(x_train , y_train , validation_data= (x_test , y_test) , epochs= 100 , initial_epoch= 20)  # Changed initial_epoch to match search epochs

Epoch 21/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7231 - loss: 0.5701 - val_accuracy: 0.7338 - val_loss: 0.5306
Epoch 22/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7345 - loss: 0.5356 - val_accuracy: 0.7338 - val_loss: 0.5137
Epoch 23/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7459 - loss: 0.5152 - val_accuracy: 0.7143 - val_loss: 0.5044
Epoch 24/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7443 - loss: 0.4999 - val_accuracy: 0.7208 - val_loss: 0.4979
Epoch 25/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7573 - loss: 0.5060 - val_accuracy: 0.7273 - val_loss: 0.4948
Epoch 26/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7704 - loss: 0.4826 - val_accuracy: 0.7532 - val_loss: 0.4918
Epoch 27/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7736 - loss: 0.4759 - val_accuracy: 0.7468 - val_loss: 0.4931
Epoch 28/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7541 - loss: 0.4860 - val_accuracy: 0